<a href="https://colab.research.google.com/github/ADRAKECROWDER/AIML2003-nlp/blob/main/week1/lab1_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
!pip install google-genai -q

from google.colab import userdata
from google import genai
import json
import re

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

This cell sets up the notebook so Gemini can run in Colab. I also added json and re. Json so it will turn the text gemini returned into structed data. re so it would clean up the json and get rid of of lines and backticks.

In [35]:
texts = [
    {
        "raw": """Providenciales: Half-Day Catamaran Sailing Trip with Snorkel.
        This 4.5-hour group excursion explores the Turks and Caicos Barrier Reef
        and includes snorkeling, a picnic lunch, and an open bar.
        From $219 per person.""",
        "expected": {
            "name": "Half-Day Catamaran Sailing Trip with Snorkel",
            "category": "snorkeling_catamaran_tour",
            "island": "Providenciales",
            "hours": 4.5,
            "price_usd": 219,
            "group_type": "group",
            "includes": ["snorkeling", "picnic lunch", "open bar"]
        }
    },
    {
        "raw": """Grace Bay: Catamaran Snorkel & Sail Adventure.
        This 2.5-hour group catamaran trip includes snorkeling
        and a premium open bar. From $129 per person.""",
        "expected": {
            "name": "Catamaran Snorkel & Sail Adventure",
            "category": "snorkeling_catamaran_tour",
            "island": "Grace Bay",
            "hours": 2.5,
            "price_usd": 129,
            "group_type": "group",
            "includes": ["snorkeling", "premium open bar"]
        }
    },
    {
        "raw": """Turks and Caicos: Iguana Islands and Mangrove Eco-Tour.
        This 3-hour group eco-tour includes pickup.
        From $145 per person.""",
        "expected": {
            "name": "Iguana Islands and Mangrove Eco-Tour",
            "category": "eco_tour",
            "island": "Turks and Caicos",
            "hours": 3,
            "price_usd": 145,
            "group_type": "group",
            "includes": ["pickup"]
        }
    },
    {
        "raw": """Providenciales: Beach and Ocean Horseback Riding Adventure.
        This 1-hour guided horseback activity includes a live tour guide.
        From $271 per person.""",
        "expected": {
            "name": "Beach and Ocean Horseback Riding Adventure",
            "category": "horseback_riding",
            "island": "Providenciales",
            "hours": 1,
            "price_usd": 271,
            "group_type": "group",
            "includes": ["live tour guide"]
        }
    },
    {
        "raw": """Turks and Caicos Islands: Private Catamaran Cruise.
        This private catamaran experience lasts 3 to 8 hours
        and includes snorkeling. From $1200 per group.""",
        "expected": {
            "name": "Private Catamaran Cruise",
            "category": "private_boat_tour",
            "island": "Turks and Caicos Islands",
            "hours": 3,
            "price_usd": 1200,
            "group_type": "private",
            "includes": ["snorkeling"]
        }
    },
    {
        "raw": """Thrilling Jet Ski Tour in Clear Blue Water of Turks & Caicos.
        This small-group guided jet ski tour lasts 1 to 2 hours.
        From $159 per person.""",
        "expected": {
            "name": "Thrilling Jet Ski Tour in Clear Blue Water of Turks & Caicos",
            "category": "jet_ski_tour",
            "island": "Turks & Caicos",
            "hours": 1,
            "price_usd": 159,
            "group_type": "small_group",
            "includes": []
        }
    },
    {
        "raw": """Luxury Lady Grace Catamaran Sunset Sailing Tour.
        This 1.5-hour group sunset sail includes a catamaran tour,
        an open bar, and gourmet hors d'oeuvres. From $169 per person.""",
        "expected": {
            "name": "Luxury Lady Grace Catamaran Sunset Sailing Tour",
            "category": "sunset_cruise",
            "island": None,
            "hours": 1.5,
            "price_usd": 169,
            "group_type": "group",
            "includes": ["catamaran tour", "open bar", "gourmet hors d'oeuvres"]
        }
    },
    {
        "raw": """Providenciales: ATV/UTV Beach Bounce Tour.
        This 2-hour guided adventure explores beach landscapes
        and sand dunes in Providenciales. From $179 per person.""",
        "expected": {
            "name": "ATV/UTV Beach Bounce Tour",
            "category": "atv_utv_tour",
            "island": "Providenciales",
            "hours": 2,
            "price_usd": 179,
            "group_type": "group",
            "includes": []
        }
    },
    {
        "raw": """Grace Bay: Parasailing Adventure Over Grace Bay.
        This 1-hour small-group parasailing activity includes pickup.
        From $90 per person.""",
        "expected": {
            "name": "Parasailing Adventure Over Grace Bay",
            "category": "parasailing",
            "island": "Grace Bay",
            "hours": 1,
            "price_usd": 90,
            "group_type": "small_group",
            "includes": ["pickup"]
        }
    },
    {
        "raw": """Providenciales: Mangrove Kayak Tour With Turtle Sightings.
        This 2-hour guided kayak tour focuses on the mangrove ecosystem.
        From $112 per person.""",
        "expected": {
            "name": "Mangrove Kayak Tour With Turtle Sightings",
            "category": "kayak_tour",
            "island": "Providenciales",
            "hours": 2,
            "price_usd": 112,
            "group_type": "group",
            "includes": []
        }
    }
]

This is the dataset for the experiment. Each example has raw travel text and the expected JSON answer.

Change I would have made was to make sure my expected data matched the raw data a little better so that my results were more accurate early on, had to fix it later.

In [36]:
def clean_json_text(text):
    text = text.strip()
    text = re.sub(r"^```json\s*", "", text)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()

This cell cleans the model output before we try to read it as JSON. I added this because the model sometimes wrapped JSON in code blocks, which caused parse errors.

In [37]:
def extract_zero_shot(text):
    response = client.models.generate_content(
        model="gemini-flash-latest",
        contents=f"Extract structured data from this Turks and Caicos activity listing as JSON.\n\n{text}"
    )
    return response.text

zero_shot_results = [extract_zero_shot(t["raw"]) for t in texts]

# Run and print raw output to see what shape it takes
for result in zero_shot_results:
    print(result)
    print("---")

```json
{
  "activity": {
    "title": "Half-Day Catamaran Sailing Trip with Snorkel",
    "location": {
      "city": "Providenciales",
      "country": "Turks and Caicos"
    },
    "duration": "4.5 hours",
    "excursion_type": "Group excursion",
    "highlights": [
      "Turks and Caicos Barrier Reef"
    ],
    "inclusions": [
      "Snorkeling",
      "Picnic lunch",
      "Open bar"
    ],
    "pricing": {
      "starting_price": 219.00,
      "currency": "USD",
      "unit": "per person"
    }
  }
}
```
---
```json
{
  "activity_name": "Catamaran Snorkel & Sail Adventure",
  "location": "Grace Bay",
  "duration": "2.5 hours",
  "group_type": "Group",
  "pricing": {
    "amount": 129,
    "currency": "USD",
    "basis": "per person",
    "starting_from": true
  },
  "inclusions": [
    "Snorkeling",
    "Premium open bar"
  ],
  "description": "This 2.5-hour group catamaran trip includes snorkeling and a premium open bar."
}
```
---
```json
{
  "activity": {
    "title": "Iguan

Zero shot: No example, no context. This runs the zero-shot baseline on the full dataset. It shows what Gemini does with almost no guidance besides the basic instruction.

In [38]:
def extract_few_shot(text):
    prompt = """Extract structured data from Turks and Caicos activity listings as JSON with these fields:
name, category, island, hours, price_usd, group_type, includes.
Respond with ONLY valid JSON on one line.

Input: "Grace Bay: Sunset sailing cruise. 1.5 hours. Group tour. Open bar included. From $169 per person."
Output: {{"name":"Sunset sailing cruise","category":"sunset_cruise","island":"Grace Bay","hours":1.5,"price_usd":169,"group_type":"group","includes":["open bar"]}}

Input: "{text}"
Output:"""
    response = client.models.generate_content(
        model="gemini-flash-latest",
        contents=prompt.format(text=text.replace('"', '\\"'))
    )
    return response.text

few_shot_results = [extract_few_shot(t["raw"]) for t in texts]

for result in few_shot_results:
    print(result)
    print("---")

{"name":"Half-Day Catamaran Sailing Trip with Snorkel","category":"catamaran_sailing","island":"Providenciales","hours":4.5,"price_usd":219,"group_type":"group","includes":["snorkeling","picnic lunch","open bar"]}
---
{"name":"Catamaran Snorkel & Sail Adventure","category":"catamaran_snorkeling","island":"Grace Bay","hours":2.5,"price_usd":129,"group_type":"group","includes":["snorkeling","premium open bar"]}
---
{"name":"Iguana Islands and Mangrove Eco-Tour","category":"eco_tour","island":"Turks and Caicos","hours":3,"price_usd":145,"group_type":"group","includes":["pickup"]}
---
{"name":"Beach and Ocean Horseback Riding Adventure","category":"horseback_riding","island":"Providenciales","hours":1,"price_usd":271,"group_type":"guided","includes":["live tour guide"]}
---
{"name":"Private Catamaran Cruise","category":"catamaran_cruise","island":"Turks and Caicos Islands","hours":"3-8","price_usd":1200,"group_type":"private","includes":["snorkeling"]}
---
{"name":"Thrilling Jet Ski Tour",

Few-shot: Prompt with one example. This runs the few-shot baseline on the full dataset. I added the line “Respond with ONLY valid JSON on one line.” so Gemini would print the JSON in a single line instead of big block paragraphs, which saved space and made the outputs easier to compare. I left zero-shot without that instruction because I wanted to see Gemini’s formatting with no guidance.

In [39]:
EXTRACTION_PROMPT = """## System Instruction
You are a data extraction specialist. Your job is to parse
unstructured Turks and Caicos activity listings into clean, consistent structured
data. You are precise about types (numbers are numbers, not
strings) and honest about missing data (use null, never guess).

## Task
Extract the following fields from the activity listing below.

## Schema
Respond with ONLY valid JSON on one line matching this exact schema:
{{
  "name": "string - activity name without a leading location prefix like 'Providenciales:' or 'Grace Bay:'",
  "category": "string - one of: snorkeling_catamaran_tour, private_boat_tour, eco_tour, horseback_riding, jet_ski_tour, sunset_cruise, atv_utv_tour, parasailing, kayak_tour",
  "island": "string - use the leading location prefix if present; otherwise null",
  "hours": "number - use the minimum when a range is given, or null if not stated",
  "price_usd": "integer - price in USD, or null if not stated",
  "group_type": "string - private, group, small_group, or null if not stated",
  "includes": ["array of short phrases copied from explicitly stated included items or services only"]
}}

## Extraction Rules
- If hours are given as a range like '1 to 2 hours' or '3 to 8 hours', use the minimum.
- If only one price is given, extract it as an integer.
- If the title starts with a location prefix followed by a colon, remove that prefix from the name and use it as island.
- Only include short phrases that are explicitly stated as included items or services.
- Do not include scenery, destinations, wildlife, or general experience descriptions in includes.
- If a field genuinely isn't present in the text, use null.
- Never fabricate a value.

## Calibration Example
Input: "Grace Bay: Parasailing Adventure. This 1-hour small-group activity includes pickup. From $90 per person."
Output: {{"name":"Parasailing Adventure","category":"parasailing","island":"Grace Bay","hours":1,"price_usd":90,"group_type":"small_group","includes":["pickup"]}}

## Activity Listing to Extract
{posting}"""

def extract_context(text):
    response = client.models.generate_content(
        model="gemini-flash-latest",
        contents=EXTRACTION_PROMPT.format(posting=text)
    )
    cleaned = clean_json_text(response.text)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {"parse_error": response.text}

context_results = [extract_context(t["raw"]) for t in texts]

for result in context_results:
    print(result)
    print("---")

{'name': 'Half-Day Catamaran Sailing Trip with Snorkel', 'category': 'snorkeling_catamaran_tour', 'island': 'Providenciales', 'hours': 4.5, 'price_usd': 219, 'group_type': 'group', 'includes': ['snorkeling', 'picnic lunch', 'open bar']}
---
{'name': 'Catamaran Snorkel & Sail Adventure', 'category': 'snorkeling_catamaran_tour', 'island': 'Grace Bay', 'hours': 2.5, 'price_usd': 129, 'group_type': 'group', 'includes': ['snorkeling', 'premium open bar']}
---
{'name': 'Iguana Islands and Mangrove Eco-Tour', 'category': 'eco_tour', 'island': 'Turks and Caicos', 'hours': 3, 'price_usd': 145, 'group_type': 'group', 'includes': ['pickup']}
---
{'name': 'Beach and Ocean Horseback Riding Adventure', 'category': 'horseback_riding', 'island': 'Providenciales', 'hours': 1, 'price_usd': 271, 'group_type': None, 'includes': ['live tour guide']}
---
{'name': 'Private Catamaran Cruise', 'category': 'private_boat_tour', 'island': 'Turks and Caicos Islands', 'hours': 3, 'price_usd': 1200, 'group_type': 'p

 Context-engineered prompt. This is a more detailed prompt that includes both examples and context instructions. Gemini has clearer rules for how to format and extract the data.

In [40]:
# Compare context-engineered results against expected values
errors = []
for t, result in zip(texts, context_results):
    expected = t["expected"]
    mismatches = []
    for key in expected:
        if key in result and result[key] != expected[key]:
            mismatches.append(f"  {key}: got {result[key]}, expected {expected[key]}")
        elif key not in result:
            mismatches.append(f"  {key}: missing from output, expected {expected[key]}")
    if mismatches:
        short = t["raw"][:80].replace("\n", " ")
        errors.append(f'Listing: "{short}..."\n' + "\n".join(mismatches))

error_report = "\n\n".join(errors) if errors else "All extractions matched."

meta_prompt = f"""You optimize prompts for structured data extraction.
Below is an extraction prompt and the fields it got wrong.

## Current Prompt
{EXTRACTION_PROMPT}

## Extraction Errors
{error_report}

## Your Task
1. Diagnose why each error occurred (type mismatch? missing
   field? wrong interpretation of the text?).
2. Propose a revised prompt that fixes these errors.
3. Return ONLY the complete revised prompt."""

refinement = client.models.generate_content(
    model="gemini-flash-latest",
    contents=meta_prompt
)
print("=== SUGGESTED REFINEMENTS ===")
print(refinement.text)

=== SUGGESTED REFINEMENTS ===
## System Instruction
You are a data extraction specialist. Your job is to parse unstructured Turks and Caicos activity listings into clean, consistent structured data. You are precise about types and logic.

## Task
Extract the following fields from the activity listing below.

## Schema
Respond with ONLY valid JSON on one line matching this exact schema:
{
  "name": "string - activity name without a leading location prefix like 'Providenciales:' or 'Grace Bay:'",
  "category": "string - one of: snorkeling_catamaran_tour, private_boat_tour, eco_tour, horseback_riding, jet_ski_tour, sunset_cruise, atv_utv_tour, parasailing, kayak_tour",
  "island": "string - extract the island or territory name (e.g., 'Providenciales', 'Grand Turk', or 'Turks & Caicos'). Prioritize the leading location prefix if present.",
  "hours": "number - use the minimum when a range is given, or null if not stated",
  "price_usd": "integer - price in USD, or null if not stated",
  "g

This cell shows Gemini where the context prompt’s output was wrong, then asks it to diagnose those mistakes and refine the prompt.

In [41]:
REFINED_PROMPT = refinement.text

def extract_refined(text):
    prompt = REFINED_PROMPT.replace("{posting}", text)
    response = client.models.generate_content(
        model="gemini-flash-latest",
        contents=prompt
    )
    cleaned = clean_json_text(response.text)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {"parse_error": response.text}

refined_results = [extract_refined(t["raw"]) for t in texts]

for result in refined_results:
    print(result)
    print("---")

{'name': 'Half-Day Catamaran Sailing Trip with Snorkel', 'category': 'snorkeling_catamaran_tour', 'island': 'Providenciales', 'hours': 4.5, 'price_usd': 219, 'group_type': 'group', 'includes': ['snorkeling', 'picnic lunch', 'open bar']}
---
{'name': 'Catamaran Snorkel & Sail Adventure', 'category': 'snorkeling_catamaran_tour', 'island': 'Grace Bay', 'hours': 2.5, 'price_usd': 129, 'group_type': 'group', 'includes': ['snorkeling', 'premium open bar']}
---
{'name': 'Iguana Islands and Mangrove Eco-Tour', 'category': 'eco_tour', 'island': 'Turks and Caicos', 'hours': 3, 'price_usd': 145, 'group_type': 'group', 'includes': ['pickup']}
---
{'name': 'Beach and Ocean Horseback Riding Adventure', 'category': 'horseback_riding', 'island': 'Providenciales', 'hours': 1, 'price_usd': 271, 'group_type': 'group', 'includes': ['live tour guide']}
---
{'name': 'Private Catamaran Cruise', 'category': 'private_boat_tour', 'island': 'Turks and Caicos Islands', 'hours': 3, 'price_usd': 1200, 'group_type':

This runs the refined prompt on the full dataset. This cell was added because the class example printed the refined prompt, but our assignment criteria asked for all 4 to be tested.

In [42]:
def parse_json_outputs(results):
    parsed = []
    for r in results:
        cleaned = clean_json_text(r)
        try:
            parsed.append(json.loads(cleaned))
        except:
            parsed.append({"parse_error": r})
    return parsed

zero_shot_parsed = parse_json_outputs(zero_shot_results)
few_shot_parsed = parse_json_outputs(few_shot_results)

print("Parse rate by approach:")
print(f"Zero-shot: {sum(1 for r in zero_shot_parsed if 'parse_error' not in r)}/{len(texts)}")
print(f"Few-shot: {sum(1 for r in few_shot_parsed if 'parse_error' not in r)}/{len(texts)}")
print(f"Context: {sum(1 for r in context_results if 'parse_error' not in r)}/{len(texts)}")
print(f"Refined: {sum(1 for r in refined_results if 'parse_error' not in r)}/{len(texts)}")

if texts[0].get("expected"):
    fields = list(texts[0]["expected"].keys())
    print(f"\n{'Field':<20} {'Zero':<10} {'Few':<10} {'Context':<10} {'Refined':<10}")
    print("-" * 60)
    for field in fields:
        zero_correct = sum(
            1 for t, r in zip(texts, zero_shot_parsed)
            if field in r and r[field] == t["expected"].get(field)
        )
        few_correct = sum(
            1 for t, r in zip(texts, few_shot_parsed)
            if field in r and r[field] == t["expected"].get(field)
        )
        context_correct = sum(
            1 for t, r in zip(texts, context_results)
            if field in r and r[field] == t["expected"].get(field)
        )
        refined_correct = sum(
            1 for t, r in zip(texts, refined_results)
            if field in r and r[field] == t["expected"].get(field)
        )
        print(f"{field:<20} {zero_correct:<10} {few_correct:<10} {context_correct:<10} {refined_correct:<10}")

Parse rate by approach:
Zero-shot: 10/10
Few-shot: 10/10
Context: 10/10
Refined: 10/10

Field                Zero       Few        Context    Refined   
------------------------------------------------------------
name                 0          9          10         10        
category             0          6          10         10        
island               0          9          9          10        
hours                0          8          10         10        
price_usd            0          10         10         10        
group_type           0          8          7          10        
includes             0          8          10         10        


This is the evaluation cell. It compares the zero-shot, few-shot, context, and refined results, shows parse rate, and shows how accurate each approach was across the fields.

I changed it so it scores each field separately instead of treating the whole JSON block as wrong if only one or two parts were off. That gave a fairer and more detailed comparison because a result could still get most of the fields right even if it missed one detail.

Reflection:
This experiment showed me that structured context helped the model stay closer to the format I wanted. Zero-shot pulled useful information, but it did not match my expected field names or structure well enough to score correctly, so it ended up with zeros across the table. Few-shot improved a lot and got most of the fields correct. The context-engineered prompt did the best overall and was stronger in several fields, especially name, category, hours, and includes. The refined prompt performed the best and got every field fully correct.

The refined prompt improved the results by making the instructions more specific and more consistent. It clarified how to handle location prefixes, how to choose the island value, how to decide group type, and what should count as an included item versus just part of the experience. It also made the rule stronger that guided tours should usually count as group unless the text clearly says private or small_group. That was useful because it fixed some of the weaker spots from the earlier prompt, especially island, group type, and includes. Those changes were more detailed than what I had in the first Context-engineered prompt.

This connects to the readings because it shows that the more clearly the task is framed, the more the model will behave/give you the output your looking for. If I did this again, I would go back and refine the expected JSON even more, because after the fact I noticed that zero-shot could have been a little more accurate if I had set it up for success better from the start. I would make sure the raw text and expected json were more closely in line with each other because you cant expect a word in the json that the raw text doesnt say.